# Macfile: Master Context File design

## Core idea

**One source file holds every context block, each tagged with `layer (what it’s about)` and `targets (where it’s allowed to go`. A “compiler” step, which can be a 50-line script or even a prompt, either packs blocks into one payload or splits them into per-platform files. The source stays the same; only the output changes.**

### 1. Four layers (this fixes the “mixed instructions” problem)

|Layer|Contains|Changes when|
|:----|:-------|:-----------|
|`agent`|Role, workflow, tone, tool-use habits of the copilot|You change copilots or working style|
|`user`|Profile, preferences, expertise level, language|You change|
|`project`|Stack, conventions, constraints, definitions of done|Project changes|
|`env`|OS, runtime, model limits, available tools|Machine or model changes|

Only `project` blocks should ever be committed to a repo. `user` and `agent` blocks stay private and travel with you.

### 2. Syntax: tagged markdown

Pseudo-XML tags wrapping plain markdown survive any tokenizer, are easy to regex-split, and LLMs parse them well (Claude is particularly good with them). Everything inside a block is ordinary markdown.

```markdown

<macfile version="0.1">
  <meta>
    name: my-master-context
    owner: jane
    updated: 2026-10-23
    default_budget_tokens: 4000
  </meta>

  <block id="agent.role" layer="agent" targets="*" load="always" priority="100">
  ## Role
  You are a senior pair-programmer. Be concise, challenge weak assumptions,
  ask at most one clarifying question before acting.
  </block>

  <block id="user.profile" layer="user" targets="web,local" load="always" priority="90">
  ## About me
  - Backend dev, Python/Go, intermediate Kubernetes
  - Prefer short answers, code first, explanation second
  </block>

  <block id="project.stack" layer="project" targets="repo,web" load="always" priority="80">
  ## Stack
  FastAPI, Postgres 16, pytest. Python 3.12, type hints required.
  </block>

  <block id="project.rules" layer="project" targets="repo,web" load="always" priority="85">
  ## Rules
  - Never edit files under /migrations by hand
  - All endpoints need tests
  </block>

  <block id="project.api-reference" layer="project" targets="repo" load="on_demand"
         src="./docs/api.md" tokens="6000">
  Pull in only when working on API routes.
  </block>

  <block id="env.local-model" layer="env" targets="local" load="always" priority="70">
  Context window 8k. Keep answers under 300 words. No tool calling available.
  </block>
</macfile>

```
**Attributes**

- `id`: dotted `layer.name`, unique, used for referencing and overrides.
- `layer`: `agent` | `user` | `projec`t | `env`.
- `targets`: where the block may appear. Use `*` for everywhere, or a list such as `web, repo, local`, or specific tags like `claude-project, custom-gpt, copilot, ollama`.
- `load`: `always` (always sent), `on_demand` (sent only if budget allows or the task needs it), or `reference` (never inlined, only linked).
- `priority`: 0 to 100. When the token budget is tight, the lowest priorities are dropped first.
- `tokens`: estimated size, so the compiler can budget.
- `src`: optional pointer to an external file, so blocks can live in separate files (modular) or inline (consolidated).
- `extends` / `overrides`: lets a project file adjust a global block without copying it.
  
### 3. Packing and splitting (the “dynamic” part)

Pack (browser/web agents): concatenate blocks that match the target, sort by `layer` then `priority`, drop `on_demand` blocks, and trim to `budget`. The result is a single markdown file to paste into Claude Projects instructions, a Custom GPT, or a chat.

**Split (local and repo workflows): emit one file per layer or target:**

|Output|	Built from|
|:-----|:-----------|
|`AGENTS.md` / `CLAUDE.md` / `.github/copilot-instructions.md`|	project blocks, plus agent blocks if you want them shared|
|Web “custom instructions” / profile field|	`user` blocks|
|Local system prompt (`Modelfile` SYSTEM, LM Studio preset)|	`agent` + `env` + trimmed `project`|
|Project knowledge uploads|	`on_demand` and `reference` blocks as separate files|

**The compiler’s only logic is a filter and a sort:**

```txt

select blocks where target in block.targets
order by layer_rank, priority desc
while total_tokens > budget: drop lowest-priority non-"always" block, then lowest "always"
render: block bodies only (strip tags for web) or keep tags (for round-tripping)

```

### 4. Design principles
1. **Single source of truth**. Never edit the generated outputs. Edit the macfile and rebuild.
2. **Separation by audience**. A block belongs to exactly one layer, so you never leak personal preferences into a shared repo.
3. **Graceful degradation**. A model that ignores the tags still reads clean markdown inside them.
4. **Declarative only**. The file describes context. It contains no executable logic, so it’s safe to share and diff.
5. **Round-trippable**. Keeping tags in the packed output lets you split it back apart later.